# Baseline Comparison: GNN vs Louvain vs node2vec

Compares GNN clustering against two standard baselines on the same TEC graph:
- **Louvain**: standard community detection, no learning
- **node2vec**: random-walk embeddings + k-means, no node features

All methods evaluated on the same metrics: silhouette score, CORUM co-complex co-assignment rate, STRING precision.

In [ ]:
!pip install python-louvain torch torch_geometric scikit-learn pandas numpy matplotlib networkx requests -q

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from torch_geometric.nn import Node2Vec
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import fisher_exact
import networkx as nx
import community as community_louvain   # python-louvain
import matplotlib.pyplot as plt
import json

from google.colab import drive
drive.mount('/content/drive')
DRIVE_PATH = '/content/drive/MyDrive/'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
# Load data
with pd.HDFStore(DRIVE_PATH + 'gene_network_data.h5') as store:
    tec = store['TEC']
np_tec_abs = np.abs(tec.to_numpy(copy=True))
gene_names = tec.index.tolist()
n_genes = len(gene_names)
gene_to_idx = {g: i for i, g in enumerate(gene_names)}
del tec
print(f'Loaded {n_genes} genes')

In [ ]:
# Build graph at threshold=0.75 (same as GNN paper)
threshold = 0.75
adj_matrix = (np_tec_abs > threshold).astype(float)
np.fill_diagonal(adj_matrix, 0)

rows, cols = np.where(np.triu(adj_matrix, k=1) > 0)
edge_index = torch.tensor(
    np.vstack([np.concatenate([rows, cols]),
               np.concatenate([cols, rows])]),
    dtype=torch.long
).to(device)

print(f'Edges: {len(rows):,}')

In [ ]:
# ── METHOD 1: Louvain community detection ────────────────────────────────────
print('Running Louvain...')
G = nx.Graph()
G.add_nodes_from(range(n_genes))
G.add_edges_from(zip(rows.tolist(), cols.tolist()))

louvain_partition = community_louvain.best_partition(G, random_state=42)
louvain_labels = np.array([louvain_partition[i] for i in range(n_genes)])
n_louvain = len(np.unique(louvain_labels))
louvain_sizes = np.bincount(louvain_labels)

print(f'Louvain: {n_louvain} communities')
print(f'Sizes (top 10): {sorted(louvain_sizes, reverse=True)[:10]}')

In [ ]:
# Louvain silhouette — use adjacency row as feature vector
# Only evaluate on the largest k communities to match GNN k=4
# Also compute on full partition
from sklearn.preprocessing import normalize

adj_feat = normalize(adj_matrix, norm='l2').astype(np.float32)

# Full Louvain partition silhouette (sampled — too many communities for full)
if n_louvain > 1:
    louvain_sil = silhouette_score(adj_feat, louvain_labels, sample_size=3000, random_state=42)
    print(f'Louvain silhouette (full partition, {n_louvain} communities): {louvain_sil:.4f}')

# Louvain collapsed to top-4 communities (fair comparison with GNN k=4)
top4_ids = np.argsort(louvain_sizes)[::-1][:4]
louvain_labels_4 = np.full(n_genes, -1)
for new_id, old_id in enumerate(top4_ids):
    louvain_labels_4[louvain_labels == old_id] = new_id
# Assign remaining genes to nearest top-4 community by majority of neighbours
for i in np.where(louvain_labels_4 == -1)[0]:
    nbr_labels = louvain_labels_4[list(G.neighbors(i))]
    nbr_labels = nbr_labels[nbr_labels >= 0]
    louvain_labels_4[i] = np.bincount(nbr_labels).argmax() if len(nbr_labels) > 0 else 0

louvain_sil_4 = silhouette_score(adj_feat, louvain_labels_4, sample_size=3000, random_state=42)
print(f'Louvain silhouette (top-4 communities): {louvain_sil_4:.4f}')

In [ ]:
# ── METHOD 2: node2vec + k-means ─────────────────────────────────────────────
print('Training node2vec...')
node2vec = Node2Vec(
    edge_index,
    embedding_dim=64,
    walk_length=20,
    context_size=10,
    walks_per_node=10,
    num_negative_samples=1,
    p=1.0, q=1.0,
    sparse=True
).to(device)

n2v_opt = torch.optim.SparseAdam(list(node2vec.parameters()), lr=0.01)
loader  = node2vec.loader(batch_size=128, shuffle=True)

node2vec.train()
for epoch in range(50):
    total_loss = 0
    for pos_rw, neg_rw in loader:
        n2v_opt.zero_grad()
        loss = node2vec.loss(pos_rw.to(device), neg_rw.to(device))
        loss.backward()
        n2v_opt.step()
        total_loss += loss.item()
    if epoch % 10 == 0:
        print(f'  Epoch {epoch}: loss={total_loss:.4f}')

node2vec.eval()
with torch.no_grad():
    n2v_emb = node2vec().cpu().numpy()
print('node2vec done.')

# k-means with k=4 to match GNN
km = KMeans(n_clusters=4, random_state=42, n_init=20)
n2v_labels = km.fit_predict(n2v_emb)
n2v_sil = silhouette_score(n2v_emb, n2v_labels, sample_size=3000, random_state=42)
print(f'node2vec silhouette (k=4): {n2v_sil:.4f}')

In [ ]:
# ── METHOD 3: Load GNN results ───────────────────────────────────────────────
gnn_emb    = np.load(DRIVE_PATH + 'noleak_gnn_embeddings.npy')
gnn_labels = np.load(DRIVE_PATH + 'noleak_gnn_cluster_labels.npy')
gnn_sil    = silhouette_score(gnn_emb, gnn_labels, sample_size=3000, random_state=42)
print(f'GNN silhouette (k=4): {gnn_sil:.4f}')
print(f'GNN cluster sizes: {np.bincount(gnn_labels)}')

In [ ]:
# ── CORUM co-complex co-assignment for all methods ───────────────────────────
corum_df = pd.read_csv(DRIVE_PATH + 'corum_allComplexes.txt', sep='\t')
org_col     = 'organism' if 'organism' in corum_df.columns else 'Organism'
corum_df    = corum_df[corum_df[org_col] == 'Human']
subunit_col = [c for c in corum_df.columns if 'subunit' in c.lower() and 'gene' in c.lower()]
name_col    = [c for c in corum_df.columns if 'complex' in c.lower() and 'name' in c.lower()]
subunit_col = subunit_col[0] if subunit_col else 'subunits(Gene name)'
name_col    = name_col[0]    if name_col    else 'ComplexName'

complexes = {}
for _, row in corum_df.iterrows():
    subunits = [g.strip() for g in str(row[subunit_col]).split(';') if g.strip()]
    if len(subunits) >= 2:
        complexes[row[name_col]] = set(subunits)

co_pairs = set()
for genes in complexes.values():
    in_data = [g for g in genes if g in gene_to_idx]
    for i in range(len(in_data)):
        for j in range(i+1, len(in_data)):
            co_pairs.add((min(in_data[i], in_data[j]), max(in_data[i], in_data[j])))

sample_pairs = list(co_pairs)[:10000]

def corum_rate(labels):
    same = sum(1 for g1, g2 in sample_pairs
               if labels[gene_to_idx[g1]] == labels[gene_to_idx[g2]])
    return same / len(sample_pairs)

def random_expected(labels):
    return sum((np.sum(labels == c) / n_genes)**2 for c in np.unique(labels))

gnn_corum    = corum_rate(gnn_labels)
n2v_corum    = corum_rate(n2v_labels)
louv_corum   = corum_rate(louvain_labels_4)
gnn_rand     = random_expected(gnn_labels)
n2v_rand     = random_expected(n2v_labels)
louv_rand    = random_expected(louvain_labels_4)

print(f'\nCORUM co-assignment rates:')
print(f'  GNN:      {gnn_corum*100:.1f}%  (random {gnn_rand*100:.1f}%,  {gnn_corum/gnn_rand:.2f}x)')
print(f'  node2vec: {n2v_corum*100:.1f}%  (random {n2v_rand*100:.1f}%,  {n2v_corum/n2v_rand:.2f}x)')
print(f'  Louvain:  {louv_corum*100:.1f}%  (random {louv_rand*100:.1f}%,  {louv_corum/louv_rand:.2f}x)')

In [ ]:
# ── Plot comparison ───────────────────────────────────────────────────────────
methods = ['GNN (GAT)', 'node2vec', 'Louvain']
sil_scores  = [gnn_sil, n2v_sil, louvain_sil_4]
corum_rates = [gnn_corum, n2v_corum, louv_corum]
rand_rates  = [gnn_rand,  n2v_rand,  louv_rand]
colors = ['#2196F3', '#FF9800', '#4CAF50']

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

bars = axes[0].bar(methods, sil_scores, color=colors, width=0.5)
axes[0].set_ylabel('Silhouette Score', fontsize=12)
axes[0].set_title('Clustering Quality (k=4)', fontsize=12)
axes[0].axhline(y=0, color='red', linestyle='--', alpha=0.5)
for bar, s in zip(bars, sil_scores):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
                 f'{s:.3f}', ha='center', fontsize=11, fontweight='bold')

x = np.arange(len(methods))
w = 0.35
axes[1].bar(x - w/2, [r*100 for r in corum_rates], w, color=colors, label='Method')
axes[1].bar(x + w/2, [r*100 for r in rand_rates],  w, color='lightgrey', label='Random expected')
axes[1].set_xticks(x)
axes[1].set_xticklabels(methods)
axes[1].set_ylabel('Co-complex pairs in same cluster (%)', fontsize=11)
axes[1].set_title('CORUM Complex Co-assignment', fontsize=12)
axes[1].legend()

plt.tight_layout()
plt.savefig('louvain_comparison_results.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: louvain_comparison_results.png')

In [ ]:
# Save results
results = {
    'GNN': {
        'n_clusters': int(len(np.unique(gnn_labels))),
        'cluster_sizes': np.bincount(gnn_labels).tolist(),
        'silhouette': gnn_sil,
        'corum_same_cluster_rate': gnn_corum,
        'corum_random_expected': gnn_rand,
        'corum_enrichment': gnn_corum / gnn_rand
    },
    'node2vec': {
        'n_clusters': 4,
        'cluster_sizes': np.bincount(n2v_labels).tolist(),
        'silhouette': n2v_sil,
        'corum_same_cluster_rate': n2v_corum,
        'corum_random_expected': n2v_rand,
        'corum_enrichment': n2v_corum / n2v_rand
    },
    'Louvain': {
        'n_clusters_full': int(n_louvain),
        'n_clusters_top4': 4,
        'silhouette_full': louvain_sil,
        'silhouette_top4': louvain_sil_4,
        'corum_same_cluster_rate': louv_corum,
        'corum_random_expected': louv_rand,
        'corum_enrichment': louv_corum / louv_rand
    }
}

with open('louvain_comparison_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print('\n=== FINAL COMPARISON ===')
for method, r in results.items():
    sil = r.get('silhouette', r.get('silhouette_top4'))
    print(f"{method}: silhouette={sil:.4f}  CORUM={r['corum_same_cluster_rate']*100:.1f}%  enrichment={r['corum_enrichment']:.2f}x")
print('Saved: louvain_comparison_results.json')